# How Math Became Manipulation

![](https://raw.githubusercontent.com/Persuasion-at-Scale/recommender-systems/main/fig/word2vec-classic-illustration.jpg)

**From library science to algorithmic persuasion in 50 years**

This notebook shows how mathematical tools for organizing documents became tools for shaping human behavior. We'll use real movie ratings and word embeddings to see how "similarity" mathematics scales from libraries to billion-user platforms.

**What you'll discover:**
- How Netflix learns your taste from movie ratings
- Why "king - man + woman = queen" changed everything
- How these same patterns decide what you see on social media

In [ ]:
# The mathematical tools that power modern persuasion
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.decomposition import TruncatedSVD  # Netflix's secret sauce
from sklearn.metrics.pairwise import cosine_similarity  # How "similar" gets computed
import requests
import zipfile
import io

plt.style.use('default')
plt.rcParams['figure.figsize'] = (10, 6)

## Part 1: How Netflix Learns Your Taste

We'll use real MovieLens data (100,000 ratings of 9,000 movies) to see how recommendation systems work. The key insight: your movie preferences form patterns that math can detect.

In [ ]:
# Download real movie rating data (MovieLens 100k)
url = "https://files.grouplens.org/datasets/movielens/ml-latest-small.zip"
response = requests.get(url)
with zipfile.ZipFile(io.BytesIO(response.content)) as zip_file:
    # Extract the files we need
    movies = pd.read_csv(zip_file.open('ml-latest-small/movies.csv'))
    ratings = pd.read_csv(zip_file.open('ml-latest-small/ratings.csv'))

print(f"📊 Real data: {len(movies):,} movies, {len(ratings):,} ratings")
print(f"Sample movies: {movies.title.head(3).tolist()}")
print(f"Rating scale: {ratings.rating.min()} to {ratings.rating.max()} stars")

In [ ]:
# Build the user-movie rating matrix (Netflix's core data structure)
rating_matrix = ratings.pivot_table(
    index='userId', 
    columns='movieId', 
    values='rating', 
    fill_value=0
)

print(f"📈 Rating matrix: {rating_matrix.shape[0]:,} users × {rating_matrix.shape[1]:,} movies")
print(f"🕳️  Sparsity: {(rating_matrix == 0).sum().sum() / rating_matrix.size:.1%} missing ratings")
print("\nThis is the fundamental challenge: most entries are unknown!")

### The Netflix Prize Solution: Matrix Factorization

**The problem:** Users rate very few movies, so the matrix is mostly empty.

**The solution:** Find hidden patterns. Maybe there are really just ~50 "types" of movies (action, romance, indie, etc.) and ~50 "types" of users (action lovers, art film fans, etc.). 

**The math:** Singular Value Decomposition finds these hidden patterns automatically.

In [ ]:
# Apply SVD to find hidden taste patterns
# This is essentially what won the Netflix Prize
n_components = 50  # Find 50 hidden "taste dimensions"
svd = TruncatedSVD(n_components=n_components, random_state=42)
user_embeddings = svd.fit_transform(rating_matrix)
movie_embeddings = svd.components_.T

print(f"🧬 Discovered {n_components} hidden taste patterns")
print(f"👥 Each user now has a {user_embeddings.shape[1]}-dimensional taste profile")
print(f"🎬 Each movie now has a {movie_embeddings.shape[1]}-dimensional genre profile")
print(f"📊 Explained variance: {svd.explained_variance_ratio_.sum():.1%}")

In [ ]:
# Find similar movies using the learned embeddings
def find_similar_movies(movie_title, top_k=5):
    # Get the movie ID
    movie_row = movies[movies.title.str.contains(movie_title, case=False)]
    if movie_row.empty:
        print(f"Movie '{movie_title}' not found!")
        return
    
    movie_id = movie_row.iloc[0].movieId
    movie_idx = list(rating_matrix.columns).index(movie_id)
    
    # Compute cosine similarity in embedding space
    movie_vec = movie_embeddings[movie_idx:movie_idx+1]
    similarities = cosine_similarity(movie_vec, movie_embeddings)[0]
    
    # Get top similar movies
    top_indices = similarities.argsort()[-top_k-1:-1][::-1]
    
    print(f"🎬 Movies similar to '{movie_row.iloc[0].title}':")
    for i, idx in enumerate(top_indices):
        similar_movie_id = rating_matrix.columns[idx]
        similar_movie = movies[movies.movieId == similar_movie_id].iloc[0]
        print(f"  {i+1}. {similar_movie.title} (similarity: {similarities[idx]:.3f})")

# Test with popular movies
find_similar_movies("Toy Story")
print()
find_similar_movies("Pulp Fiction")
print()
find_similar_movies("Titanic")

### Visualizing the Movie Taste Space

Let's see what the algorithm learned by plotting movies in 2D space. Movies close together should have similar appeal.

In [ ]:
# Plot movies in the first 2 dimensions of taste space
plt.figure(figsize=(12, 8))

# Get popular movies for plotting (movies with many ratings)
movie_popularity = ratings.groupby('movieId').size()
popular_movies = movie_popularity[movie_popularity >= 100].index

# Get coordinates for popular movies
popular_indices = [list(rating_matrix.columns).index(mid) for mid in popular_movies if mid in rating_matrix.columns]
popular_coords = movie_embeddings[popular_indices]

# Plot the movies
plt.scatter(popular_coords[:, 0], popular_coords[:, 1], alpha=0.6, s=60)

# Label some famous movies
famous_titles = ['Toy Story (1995)', 'Pulp Fiction (1994)', 'Titanic (1997)', 
                'Star Wars (1977)', 'Forrest Gump (1994)', 'The Matrix (1999)']

for title in famous_titles:
    movie_row = movies[movies.title == title]
    if not movie_row.empty and movie_row.iloc[0].movieId in rating_matrix.columns:
        movie_idx = list(rating_matrix.columns).index(movie_row.iloc[0].movieId)
        x, y = movie_embeddings[movie_idx, 0], movie_embeddings[movie_idx, 1]
        plt.annotate(title.replace(' (1995)', '').replace(' (1994)', '').replace(' (1997)', '').replace(' (1977)', '').replace(' (1999)', ''), 
                    (x, y), xytext=(5, 5), textcoords='offset points', fontsize=9, 
                    bbox=dict(boxstyle='round,pad=0.3', facecolor='yellow', alpha=0.7))

plt.xlabel('Taste Dimension 1 (e.g., Action ↔ Drama)')
plt.ylabel('Taste Dimension 2 (e.g., Mainstream ↔ Indie)')
plt.title('Movies in Latent Taste Space\n(Close = Similar Appeal)')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("💡 Key insight: Mathematical 'distance' = perceptual 'similarity'")
print("📏 Small distance in this space = similar user ratings = similar appeal")

## Part 2: When Math Learned Human Meaning

The Netflix approach worked on movie ratings. But in 2013, Google created Word2Vec - the same mathematics applied to language itself. This changed everything.

**The breakthrough:** Words with similar meanings appear in similar contexts. So we can learn word meanings just from text.

In [ ]:
# Load ACTUAL pre-trained word embeddings
# These are real vectors trained on billions of words, showing genuine learned relationships

try:
    # Try to load real GloVe embeddings
    import gensim.downloader as api
    print("📥 Downloading real GloVe embeddings (50MB, may take 1-2 minutes first time)...")
    
    # Load smaller GloVe model: 50-dimensional vectors trained on 6B tokens
    model = api.load("glove-wiki-gigaword-50")
    print("✅ Loaded real word embeddings trained on Wikipedia + Gigaword!")
    
    # Test which words are available in the vocabulary
    test_words = ['king', 'queen', 'man', 'woman', 'programmer', 'homemaker', 
                  'doctor', 'nurse', 'paris', 'france', 'berlin', 'germany',
                  'good', 'evil', 'he', 'she', 'prince', 'princess']
    
    available_words = [word for word in test_words if word in model.key_to_index]
    print(f"📝 Found {len(available_words)} words in vocabulary: {available_words}")
    
    # Extract the ACTUAL vectors learned from text
    word_vectors = {}
    for word in available_words:
        word_vectors[word] = model[word]
    
    print(f"🧮 Using real {model.vector_size}-dimensional embeddings")
    print("🔍 These show genuine patterns learned from billions of words of text")
    
except ImportError:
    print("⚠️  gensim not available, installing...")
    import subprocess
    subprocess.check_call(['pip', 'install', 'gensim'])
    import gensim.downloader as api
    model = api.load("glove-wiki-gigaword-50")
    available_words = [word for word in test_words if word in model.key_to_index]
    word_vectors = {word: model[word] for word in available_words}
    
except Exception as e:
    print(f"⚠️  Could not download embeddings: {e}")
    print("📚 This would normally use real pre-trained vectors")
    print("💡 In a working environment, you'd see actual learned relationships")
    # Provide a minimal fallback that explains what would happen
    word_vectors = {}

def vector_analogy(a, b, c, word_vectors):
    """Solve analogies using real vector arithmetic: a is to b as c is to ?"""
    if not all(word in word_vectors for word in [a, b, c]):
        missing = [w for w in [a, b, c] if w not in word_vectors]
        print(f"⚠️  Missing words in vocabulary: {missing}")
        return None, 0.0
        
    # The famous formula discovered by Mikolov: king - man + woman ≈ queen
    target_vector = word_vectors[a] - word_vectors[b] + word_vectors[c]
    
    # Find the closest word using cosine similarity
    best_word = None
    best_similarity = -1
    
    for word, vector in word_vectors.items():
        if word in [a, b, c]:  # Skip input words
            continue
            
        # Compute cosine similarity
        dot_product = np.dot(target_vector, vector)
        norms = np.linalg.norm(target_vector) * np.linalg.norm(vector)
        similarity = dot_product / norms if norms > 0 else 0
        
        if similarity > best_similarity:
            best_similarity = similarity
            best_word = word
    
    return best_word, best_similarity

if word_vectors:
    print("\n🔥 Testing the Word2Vec breakthrough with REAL embeddings:")
    print("Mathematical operations on learned word meanings!\n")

    # Test the famous examples if words are available
    examples = [
        ('king', 'man', 'woman', 'Should find: queen'),
        ('paris', 'france', 'germany', 'Should find: berlin'),
        ('good', 'man', 'woman', 'Tests gender associations'),
        ('he', 'man', 'woman', 'Should find: she')
    ]
    
    for a, b, c, expected in examples:
        result, sim = vector_analogy(a, b, c, word_vectors)
        if result:
            print(f"📐 {a} - {b} + {c} = {result} (similarity: {sim:.3f}) | {expected}")
        else:
            print(f"❌ Cannot test {a} - {b} + {c}: missing words")
    
    print(f"\n💡 These relationships emerged naturally from analyzing text")
    print("⚠️  Real embeddings reveal both useful patterns AND human biases")
    print("📊 Same mathematics now powers TikTok, Facebook, YouTube recommendations")
else:
    print("\n📚 Real word embeddings would show:")
    print("👑 king - man + woman ≈ queen (royal relationships)")
    print("🌍 paris - france + germany ≈ berlin (geographic patterns)")  
    print("👨‍💻 he - man + woman ≈ she (gender patterns)")
    print("⚠️  Including problematic biases like 'man:programmer :: woman:homemaker'")
    print("\n🔍 These emerge from training on human-written text")

In [ ]:
# Visualize the bias in 2D
from sklearn.decomposition import PCA

# Get all word vectors
words = list(word_vectors.keys())
vectors = np.array(list(word_vectors.values()))

# Reduce to 2D for plotting
pca = PCA(n_components=2)
coords_2d = pca.fit_transform(vectors)

plt.figure(figsize=(12, 8))

# Plot word positions
for i, word in enumerate(words):
    x, y = coords_2d[i]
    
    # Color-code by category
    if word in ['king', 'queen']:
        color = 'gold'
        marker = '*'
        size = 200
    elif word in ['man', 'woman']:
        color = 'red'
        marker = 'o'
        size = 150
    elif word in ['programmer', 'homemaker', 'doctor', 'nurse']:
        color = 'blue'
        marker = 's'
        size = 100
    else:  # countries/cities
        color = 'green'
        marker = '^'
        size = 100
    
    plt.scatter(x, y, c=color, marker=marker, s=size, alpha=0.8)
    plt.annotate(word, (x, y), xytext=(5, 5), textcoords='offset points', 
                fontsize=12, fontweight='bold')

# Draw the gender direction arrow
man_coord = coords_2d[words.index('man')]
woman_coord = coords_2d[words.index('woman')]
plt.annotate('', xy=woman_coord, xytext=man_coord, 
            arrowprops=dict(arrowstyle='->', lw=3, color='red', alpha=0.7))
plt.text((man_coord[0] + woman_coord[0])/2, (man_coord[1] + woman_coord[1])/2 + 0.1, 
         'Gender Direction', fontsize=10, ha='center', color='red')

plt.title('Word Embeddings Learned Human Biases\n("man:programmer :: woman:homemaker")', fontsize=14)
plt.xlabel('Embedding Dimension 1')
plt.ylabel('Embedding Dimension 2')
plt.grid(True, alpha=0.3)

# Add legend
plt.scatter([], [], c='gold', marker='*', s=200, label='Royalty', alpha=0.8)
plt.scatter([], [], c='red', marker='o', s=150, label='Gender', alpha=0.8)
plt.scatter([], [], c='blue', marker='s', s=100, label='Professions', alpha=0.8)
plt.scatter([], [], c='green', marker='^', s=100, label='Geography', alpha=0.8)
plt.legend()

plt.tight_layout()
plt.show()

print("💡 Key insight: Embeddings that capture meaning also capture prejudice")
print("⚖️  When AI learns from human text, it learns human biases too")

## The Connection: From Libraries to Manipulation

**What we've seen:**

1. **Netflix (2009):** Math finds patterns in movie ratings → personalized recommendations
2. **Word2Vec (2013):** Same math finds patterns in language → semantic understanding
3. **Today:** These patterns decide what billions see on social media

**The progression:**
- 📚 **1970s:** TF-IDF organized library documents
- 🎬 **2009:** Matrix factorization predicted movie preferences 
- 📝 **2013:** Word embeddings captured human meaning (and bias)
- 📱 **2020s:** Same mathematics powers TikTok, Facebook, YouTube

**The transformation:** Tools for understanding became tools for persuasion.

Mathematics doesn't care whether it's organizing a library or shaping political opinions. The same "similarity = small distance" principle that helps you find books now decides what information reaches your brain.

**The scale:** What started with thousands of documents now operates on billions of users, optimizing not just for relevance, but for engagement, attention, and influence.

---

*Next: How do large language models generate content designed to change minds?*